**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Model Compression & Edge AI

Trained models are fat with redundancy; deployment targets are small. Three sessions on the compression toolkit — quantization (closing the loop with [FPGA fixed-point](../../Intro_FPGA/Intro_FPGA.ipynb)!), pruning, and distillation — each measured, not asserted, on a model we train in-notebook.

## 1. Pre-requisites

- [Intro to CNN](../Intro_CNN/Intro_CNN.ipynb) — we reuse its spectrogram classifier setup.
- [Scaling Neural Networks](../Scale_NN/Scale_NN.ipynb) — the accounting mindset.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy import signal as sig
torch.manual_seed(0); rng = np.random.default_rng(0)

# a HARDER 6-class cousin of the CNN workshop task, at low SNR — so trade-offs are visible
fs, T_dur = 1024, 1.0
t = np.arange(0, T_dur, 1/fs)
def make_example(cls):
    f0 = rng.uniform(60, 200)
    if cls == 0:   x = sig.chirp(t, f0=f0, f1=f0+150, t1=T_dur)            # chirp up
    elif cls == 1: x = sig.chirp(t, f0=f0+150, f1=f0, t1=T_dur)            # chirp down
    elif cls == 2: x = np.sin(2*np.pi*f0*t)                                 # tone
    elif cls == 3: x = np.sin(2*np.pi*f0*t) + np.sin(2*np.pi*(f0+90)*t)     # two tones
    elif cls == 4: x = np.sin(2*np.pi*f0*t) * (1 + np.sin(2*np.pi*4*t))     # AM tone
    else:
        x = np.zeros_like(t); s = rng.integers(0, len(t)//2); x[s:s+len(t)//3] = rng.standard_normal(len(t)//3)
    x = x + 1.2*rng.standard_normal(len(t))                                 # LOW SNR
    _, _, S = sig.stft(x, fs=fs, nperseg=64)
    S = np.log1p(np.abs(S))[:32, :32]
    return (S - S.mean()) / (S.std() + 1e-6)

n_cls = 6
X = torch.from_numpy(np.stack([make_example(c % n_cls) for c in range(1500)]).astype(np.float32)).unsqueeze(1)
y = torch.tensor([c % n_cls for c in range(1500)])
tr, te = torch.arange(1000), torch.arange(1000, 1500)

def accuracy(m):
    m.eval()
    with torch.no_grad():
        return (m(X[te]).argmax(1) == y[te]).float().mean().item()

---
### 🕐 Session 1 of 3 — *Quantization* (~40 min)
**Goal:** shrink weights from float32 to int8; measure the size/accuracy trade.
**Builds on:** [CNN](../Intro_CNN/Intro_CNN.ipynb); [Scale_NN](../Scale_NN/Scale_NN.ipynb). &nbsp; **Feeds into:** Session 2 (pruning).

---

## 2. Fewer Bits Per Weight

💡 **Intuition.** Weights are stored as float32 out of habit, not need: after training, each layer's weights occupy a narrow range that **8-bit integers** cover with plenty of resolution. Quantization stores `int8 + scale` per tensor — 4× smaller, and on supporting hardware, faster (integer math, less memory traffic). It is exactly the [Q1.15 fixed-point story](../../Intro_FPGA/Intro_FPGA.ipynb) from FPGA land: the same signal-to-quantization-noise arithmetic, applied to weights.

In [2]:
class SpecCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classify = nn.Sequential(nn.Flatten(), nn.Linear(32*8*8, 64), nn.ReLU(), nn.Linear(64, 6))
    def forward(self, x): return self.classify(self.features(x))

model = SpecCNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()
for ep in range(5):
    model.train()
    for i in range(0, 1000, 64):
        opt.zero_grad(); lossf(model(X[tr[i:i+64]]), y[tr[i:i+64]]).backward(); opt.step()
acc_fp32 = accuracy(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"fp32 baseline: {acc_fp32:.1%} accuracy, {n_params*4/1024:.0f} KB")

fp32 baseline: 99.8% accuracy, 533 KB


In [3]:
# post-training quantization by hand — per-tensor symmetric int8
def quantize_model(model, bits=8):
    qmax = 2**(bits-1) - 1
    state = {}
    for name, p in model.state_dict().items():
        scale = p.abs().max() / qmax if p.abs().max() > 0 else 1.0
        q = torch.clamp((p / scale).round(), -qmax-1, qmax)
        state[name] = (q.to(torch.int8 if bits <= 8 else torch.int32), scale)
    return state

def dequantize_into(model, qstate):
    sd = {name: q.float() * s for name, (q, s) in qstate.items()}
    model.load_state_dict(sd)

for bits in [8, 4, 2]:
    m2 = SpecCNN(); qs = quantize_model(model, bits)
    dequantize_into(m2, qs)
    print(f"int{bits}: {accuracy(m2):.1%} accuracy, {n_params*bits/8/1024:.0f} KB  "
          f"({32//bits}x smaller)")

int8: 99.8% accuracy, 133 KB  (4x smaller)
int4: 99.8% accuracy, 67 KB  (8x smaller)
int2: 83.4% accuracy, 33 KB  (16x smaller)


Read the table: int8 — and here even int4 — is free; int2 collapses. Where the cliff sits depends on the model and task; measuring it (as you just did) is the professional habit. Modern LLM deployment lives at 4–8 bits with cleverer schemes (per-channel scales, GPTQ/AWQ-style calibration) that push the cliff further out.

---
### 🕐 Session 2 of 3 — *Pruning* (~35 min)
**Goal:** remove small weights entirely; discover how much of the network was never needed.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (distillation).

---

## 3. Fewer Weights, Period

💡 **Intuition.** Trained networks are over-provisioned: many weights end up tiny and removable. **Magnitude pruning** zeroes the smallest p%, then (crucially) *fine-tunes* the survivors to absorb the loss. Unstructured sparsity shrinks storage; hardware speedups need *structured* pruning (whole channels/heads) so dense math stays dense. The deeper lesson is the lottery-ticket observation: a small subnetwork was doing most of the work all along.

In [4]:
import copy
def prune_magnitude(model, frac):
    m2 = copy.deepcopy(model)
    with torch.no_grad():
        all_w = torch.cat([p.abs().flatten() for p in m2.parameters() if p.dim() > 1])
        thresh = torch.quantile(all_w, frac)
        masks = []
        for p in m2.parameters():
            if p.dim() > 1:
                mask = (p.abs() > thresh).float()
                p.mul_(mask); masks.append(mask)
    return m2, masks

print("prune → accuracy (no fine-tune) → after 1 epoch of fine-tune:")
for frac in [0.5, 0.8, 0.95]:
    m2, masks = prune_magnitude(model, frac)
    a0 = accuracy(m2)
    opt2 = torch.optim.Adam(m2.parameters(), lr=5e-4)
    m2.train()
    for i in range(0, 1000, 64):
        opt2.zero_grad(); lossf(m2(X[tr[i:i+64]]), y[tr[i:i+64]]).backward(); opt2.step()
        with torch.no_grad():                      # re-apply masks: pruned stays pruned
            mi = 0
            for p in m2.parameters():
                if p.dim() > 1: p.mul_(masks[mi]); mi += 1
    print(f"  {frac:.0%} pruned: {a0:.1%} → {accuracy(m2):.1%}")

prune → accuracy (no fine-tune) → after 1 epoch of fine-tune:
  50% pruned: 99.8% → 99.6%
  80% pruned: 99.8% → 99.6%


  95% pruned: 91.8% → 97.0%


---
### 🕐 Session 3 of 3 — *Knowledge Distillation* (~40 min)
**Goal:** train a small student to mimic the big teacher's soft predictions.
**Builds on:** Session 2.

---

## 4. Teaching a Smaller Model

💡 **Intuition.** A trained teacher knows more than its labels: its **soft probabilities** encode which classes are *almost* confusable — 'this chirp is 80% chirp, 19% tone' is a richer target than 'chirp'. Distillation trains a small student against those softened outputs (temperature > 1 amplifies the dark knowledge) plus the true labels. The student often lands closer to the teacher than the same architecture trained on labels alone.

In [5]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(4))
        self.classify = nn.Sequential(nn.Flatten(), nn.Linear(4*8*8, 6))
    def forward(self, x): return self.classify(self.features(x))

def train_student(distill, epochs=8, T_=3.0, alpha=0.7):
    torch.manual_seed(1)
    s = TinyCNN()
    opt_s = torch.optim.Adam(s.parameters(), lr=2e-3)
    kl = nn.KLDivLoss(reduction="batchmean")
    for ep in range(epochs):
        s.train()
        for i in range(0, 1000, 64):
            xb, yb = X[tr[i:i+64]], y[tr[i:i+64]]
            logits_s = s(xb)
            loss = lossf(logits_s, yb)
            if distill:
                with torch.no_grad():
                    logits_t = model(xb)
                soft = kl(torch.log_softmax(logits_s/T_, 1), torch.softmax(logits_t/T_, 1)) * T_*T_
                loss = alpha*soft + (1-alpha)*loss
            opt_s.zero_grad(); loss.backward(); opt_s.step()
    return s

s_plain = train_student(False)
s_dist  = train_student(True)
n_tiny = sum(p.numel() for p in TinyCNN().parameters())
print(f"teacher ({n_params/1000:.0f}k params): {acc_fp32:.1%}")
print(f"student ({n_tiny/1000:.1f}k params) trained on labels:      {accuracy(s_plain):.1%}")
print(f"student ({n_tiny/1000:.1f}k params) distilled from teacher: {accuracy(s_dist):.1%}")

teacher (136k params): 99.8%
student (1.6k params) trained on labels:      97.0%
student (1.6k params) distilled from teacher: 97.8%


## 5. Conclusion

Quantize to 8 bits for free, prune what training over-provisioned, distill what remains into a smaller architecture — then stack all three for edge deployment. Every technique is the same trade, measured: bits and weights for accuracy, priced explicitly.

---
## Where next

- [Intro to FPGA](../../Intro_FPGA/Intro_FPGA.ipynb) — where the int8 weights land in silicon.
- [Scaling Neural Networks](../Scale_NN/Scale_NN.ipynb) — compression's mirror image.
- [LLMs from the Ground Up](../LLMs_from_the_Ground_Up.ipynb) — why 4-bit LLMs are everywhere.